# 01 — Exploratory Data Analysis

**Goal**: Understand the synthetic crop sensor dataset, identify patterns,
detect anomalies, and establish baselines for feature engineering.

**Crops**: banana, maize, cacao, rice
**Features**: temperature, humidity, soil moisture, rainfall
**Target**: yield (kg/ha)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="viridis")
%matplotlib inline

print("Libraries loaded successfully.")

## 1. Load Synthetic Data

Generate synthetic crop data for EDA. In production, this would load from
TimescaleDB via the backend API.

In [ ]:
np.random.seed(42)
n_samples = 10000

crop_types = ['banana', 'maize', 'cacao', 'rice']

data = {
    'crop_type': np.random.choice(crop_types, n_samples),
    'temp_mean': np.random.normal(25, 5, n_samples),
    'temp_max': np.random.normal(32, 4, n_samples),
    'temp_min': np.random.normal(18, 3, n_samples),
    'humidity_mean': np.random.normal(75, 10, n_samples),
    'soil_moisture_mean': np.random.normal(55, 15, n_samples),
    'rain_total': np.random.exponential(50, n_samples),
    'days_since_planting': np.random.randint(0, 365, n_samples),
    'area_ha': np.random.uniform(0.5, 20, n_samples),
}

df = pd.DataFrame(data)

# Generate synthetic yield targets (with noise)
base_yields = {'banana': 35000, 'maize': 6000, 'cacao': 800, 'rice': 4500}
df['yield_kg_ha'] = df['crop_type'].map(base_yields)
df['yield_kg_ha'] += np.random.normal(0, df['yield_kg_ha'] * 0.15, n_samples)
df['yield_kg_ha'] = df['yield_kg_ha'].clip(lower=0)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Summary Statistics

In [ ]:
df.describe()

In [ ]:
# Per crop type statistics
df.groupby('crop_type')['yield_kg_ha'].describe()

## 3. Missing Value Analysis

Check for missing values and understand data completeness patterns.

In [ ]:
# Introduce some missing values for realistic analysis
df_with_na = df.copy()
for col in ['temp_mean', 'humidity_mean', 'soil_moisture_mean', 'rain_total']:
    mask = np.random.random(n_samples) < 0.05
    df_with_na.loc[mask, col] = np.nan

missing = df_with_na.isnull().sum()
missing_pct = (missing / len(df_with_na)) * 100
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

## 4. Distribution of Target Variable

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, crop in enumerate(crop_types):
    ax = axes[idx // 2, idx % 2]
    subset = df[df['crop_type'] == crop]['yield_kg_ha']
    ax.hist(subset, bins=30, alpha=0.7, edgecolor='black')
    ax.set_title(f'{crop.capitalize()} — Yield Distribution')
    ax.set_xlabel('Yield (kg/ha)')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 5. Correlations with Target

In [ ]:
# One-hot encode crop type for correlation analysis
df_encoded = pd.get_dummies(df, columns=['crop_type'], prefix='crop')
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns
corr_matrix = df_encoded[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with yield
target_corr = corr_matrix['yield_kg_ha'].drop('yield_kg_ha').sort_values(ascending=False)
plt.figure(figsize=(10, 6))
target_corr.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Correlation with Yield (kg/ha)')
plt.ylabel('Pearson Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print("\nCorrelation values:")
print(target_corr)

## 6. Key Findings

1. **Crop type dominates yield variance** — different crops have vastly different
   yield magnitudes (banana ~35 t/ha vs cacao ~800 kg/ha).
2. **Temperature and days_since_planting** show moderate correlation with yield.
3. **Rainfall and soil moisture** have weaker linear correlation but likely
   interact non-linearly (interaction effects).
4. **Missing data is ~5%** across features — imputation will be needed.
5. **Yield distributions** are roughly normal per crop type, suggesting
   MSE/R² are appropriate evaluation metrics.

**Next**: Proceed to [02 Feature Engineering](02_feature_engineering.ipynb)